# RF Spectrogram CNN Backbone — Resolution Sweep, Benchmarking, Metrics, Ablation
This notebook standardizes experiments and writes orderly artifacts under `runs/`.
It auto-discovers class folders from `DATADIR` (no hardcoded labels).

In [ ]:
# =========================
# 0) Imports
# =========================
import os, json, time, platform, socket, uuid
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn import metrics

import tensorflow as tf
from tensorflow.keras import layers, Model


In [ ]:
# =========================
# 1) Dataset Path + Auto Class Discovery
# =========================
# NOTE: Set this to your dataset root that contains one subfolder per class.
DATADIR = Path("/home/bhumikas/DronesRF/DroneRFb_Spectra/Condata/")

if not DATADIR.exists():
    raise FileNotFoundError(f"DATADIR does not exist: {DATADIR}")

CATEGORIES = sorted([p.name for p in DATADIR.iterdir() if p.is_dir()])
NUM_CLASSES = len(CATEGORIES)

if NUM_CLASSES < 2:
    raise RuntimeError(f"Need at least 2 class folders under {DATADIR}. Found: {NUM_CLASSES}")

print("Detected classes:", NUM_CLASSES)
print("First classes:", CATEGORIES[:10])


In [ ]:
# =========================
# 2) Reproducibility
# =========================
SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)


In [ ]:
# =========================
# 3) Utility: Runs directory + JSON/CSV helpers + Device logging
# =========================
RUNS_DIR = Path("runs")
RUNS_DIR.mkdir(exist_ok=True)

def now_run_id(prefix="run"):
    return f"{prefix}_{time.strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

def get_device_info():
    return {
        "hostname": socket.gethostname(),
        "platform": platform.platform(),
        "python_version": platform.python_version(),
        "tf_version": tf.__version__,
        "physical_devices": [d.name for d in tf.config.list_physical_devices()],
        "gpus": [d.name for d in tf.config.list_physical_devices("GPU")],
        "cpus": [d.name for d in tf.config.list_physical_devices("CPU")],
    }

def save_json(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def append_row_csv(csv_path: Path, row_dict: dict):
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame([row_dict])
    if csv_path.exists():
        old = pd.read_csv(csv_path)
        df = pd.concat([old, df], ignore_index=True)
    df.to_csv(csv_path, index=False)


In [ ]:
# =========================
# 4) Robust Image Loader (fails fast for missing folders)
# =========================
def load_images_from_folders(datadir: Path, categories, img_size=(64,64), max_images_per_class=None):
    X, Y = [], []
    for class_idx, cat in enumerate(categories):
        class_dir = datadir / cat
        if not class_dir.exists():
            raise FileNotFoundError(f"Missing class folder: {class_dir}")

        files = sorted([p for p in class_dir.iterdir() if p.is_file()])
        if max_images_per_class is not None:
            files = files[:max_images_per_class]

        for p in files:
            img = cv2.imread(str(p), cv2.IMREAD_COLOR)
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, img_size, interpolation=cv2.INTER_AREA)
            X.append(img)
            Y.append(class_idx)

    if len(X) == 0:
        raise RuntimeError("No images loaded. Check DATADIR and folder contents.")

    X = np.asarray(X, dtype=np.float32) / 255.0
    Y = np.asarray(Y, dtype=np.int64)
    return X, Y


In [ ]:
# =========================
# 5) Model: Custom DenseNet-inspired (ablation-ready: growth_rate, compression, depth)
# =========================
def create_custom_densenet(input_shape, num_classes, growth_rate=8, compression=0.5, depth=(3,3,3)):
    inputs = tf.keras.Input(shape=input_shape)

    x = layers.BatchNormalization()(inputs)
    x = layers.Conv2D(growth_rate * 2, kernel_size=3, padding='same', activation='relu')(x)

    def dense_block(x, num_layers, growth_rate):
        for _ in range(num_layers):
            out = layers.BatchNormalization()(x)
            out = layers.Conv2D(growth_rate, kernel_size=3, padding='same', activation='relu')(out)
            x = layers.Concatenate()([x, out])
        return x

    def transition_block(x, compression):
        ch = int(x.shape[-1])
        reduced_filters = max(1, int(ch * compression))
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(reduced_filters, kernel_size=1, padding='same', activation='relu')(x)
        x = layers.AveragePooling2D(pool_size=2)(x)
        return x

    for num_layers in depth:
        x = dense_block(x, num_layers, growth_rate)
        x = transition_block(x, compression)

    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return Model(inputs, outputs)


In [ ]:
# =========================
# 6) Metrics + Confusion Matrix (JSON + CSV)
# =========================
def compute_metrics(y_true, y_pred, labels):
    acc = metrics.accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = metrics.precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    cm = metrics.confusion_matrix(y_true, y_pred, labels=np.arange(len(labels)))

    metrics_out = {
        "accuracy": float(acc),
        "macro_precision": float(prec),
        "macro_recall": float(rec),
        "macro_f1": float(f1),
        "support": int(len(y_true)),
    }
    cm_json = {"labels": labels, "matrix": cm.tolist()}
    return metrics_out, cm_json


In [ ]:
# =========================
# 7) Inference Benchmarking (batch=1, warmup + timed runs)
# =========================
@tf.function
def _infer_step(model, x):
    return model(x, training=False)

def benchmark_inference(model, input_shape, warmup=30, runs=200):
    x = tf.random.uniform((1, *input_shape), dtype=tf.float32)

    # warmup
    for _ in range(warmup):
        _ = _infer_step(model, x)

    # timed loop
    t0 = time.perf_counter()
    for _ in range(runs):
        _ = _infer_step(model, x)
    t1 = time.perf_counter()

    total_s = (t1 - t0)
    avg_s = total_s / runs
    fps = 1.0 / avg_s if avg_s > 0 else float("inf")

    # per-run stats
    times = []
    for _ in range(min(runs, 200)):
        a = time.perf_counter()
        _ = _infer_step(model, x)
        b = time.perf_counter()
        times.append(b - a)

    times = np.array(times, dtype=np.float64)

    return {
        "warmup_runs": int(warmup),
        "timed_runs": int(runs),
        "avg_latency_ms": float(avg_s * 1e3),
        "fps": float(fps),
        "p50_latency_ms": float(np.percentile(times, 50) * 1e3),
        "p90_latency_ms": float(np.percentile(times, 90) * 1e3),
        "p95_latency_ms": float(np.percentile(times, 95) * 1e3),
        "p99_latency_ms": float(np.percentile(times, 99) * 1e3),
        "min_latency_ms": float(times.min() * 1e3),
        "max_latency_ms": float(times.max() * 1e3),
    }


In [ ]:
# =========================
# 8) Single Run: Train + Eval + Save Artifacts to runs/<run_id>/
# =========================
def train_eval_one_run(
    datadir: Path,
    categories,
    img_res: int,
    growth_rate: int,
    compression: float,
    depth: tuple,
    epochs=30,
    batch_size=32,
    max_images_per_class=None,
    test_size=0.15,
    val_size=0.15,
    seed=42
):
    run_id = now_run_id(prefix=f"res{img_res}_k{growth_rate}_c{compression}_d{'-'.join(map(str,depth))}")
    run_dir = RUNS_DIR / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    config = {
        "img_res": img_res,
        "growth_rate": growth_rate,
        "compression": compression,
        "depth": list(depth),
        "epochs": epochs,
        "batch_size": batch_size,
        "max_images_per_class": max_images_per_class,
        "split": {"train": 1.0 - test_size - val_size, "val": val_size, "test": test_size},
        "seed": seed,
        "num_classes": len(categories),
        "labels": categories,
    }
    save_json(run_dir / "config.json", config)
    save_json(run_dir / "device.json", get_device_info())

    # data
    X, Y = load_images_from_folders(datadir, categories, img_size=(img_res, img_res), max_images_per_class=max_images_per_class)

    # split (stratified + seeded)
    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X, Y, test_size=(test_size + val_size), random_state=seed, stratify=Y
    )
    rel_val = val_size / (test_size + val_size)
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=(1.0 - rel_val), random_state=seed, stratify=y_tmp
    )

    # model
    model = create_custom_densenet(
        input_shape=(img_res, img_res, 3),
        num_classes=len(categories),
        growth_rate=growth_rate,
        compression=compression,
        depth=depth
    )

    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3, clipvalue=1.0)
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    callbacks = [
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1
    )

    # eval
    y_prob = model.predict(X_test, batch_size=64, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    metrics_out, cm_json = compute_metrics(y_test, y_pred, labels=categories)
    save_json(run_dir / "metrics.json", metrics_out)
    save_json(run_dir / "confusion_matrix.json", cm_json)

    cm = np.array(cm_json["matrix"], dtype=int)
    cm_df = pd.DataFrame(cm, index=categories, columns=categories)
    cm_df.to_csv(run_dir / "confusion_matrix.csv", index=True)

    bench = benchmark_inference(model, input_shape=(img_res, img_res, 3), warmup=30, runs=200)
    save_json(run_dir / "latency.json", bench)

    # aggregate CSV row
    aggregate_row = {
        "run_id": run_id,
        "img_res": img_res,
        "growth_rate": growth_rate,
        "compression": compression,
        "depth": "-".join(map(str, depth)),
        "epochs": epochs,
        "batch_size": batch_size,
        "max_images_per_class": max_images_per_class if max_images_per_class is not None else -1,
        **metrics_out,
        **bench,
    }
    append_row_csv(RUNS_DIR / "_aggregate_metrics.csv", aggregate_row)

    # optional: save training history
    hist_df = pd.DataFrame(history.history)
    hist_df.to_csv(run_dir / "history.csv", index=False)

    return run_dir, aggregate_row, history


In [ ]:
# =========================
# 9) Experiment Launcher
# =========================
BASE_EPOCHS = 30
BASE_BATCH  = 32
MAX_PER_CLASS = 200   # set None to use all images

# Baseline architecture for sweeps
baseline_params = dict(growth_rate=8, compression=0.5, depth=(3,3,3))


In [ ]:
# =========================
# 10) Plotting Functions for Ablation Study
# =========================
def plot_ablation_accuracy_loss(ablation_histories, save_dir=None):
    """
    Plot accuracy and loss graphs for ablation study results.
    
    Args:
        ablation_histories: List of dicts with 'history', 'growth_rate', 'compression', 'depth', 'batch_size'
        save_dir: Optional directory to save plots
    """
    if save_dir:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
    
    # Group by ablation type (using baseline values for filtering)
    baseline_comp = 0.5
    baseline_depth = (3,3,3)
    baseline_gr = 8
    
    # Growth rate runs: filter by baseline compression and depth
    growth_rate_runs = [h for h in ablation_histories if h["compression"] == baseline_comp and h["depth"] == baseline_depth]
    
    # Compression runs: filter by baseline growth_rate and depth
    compression_runs = [h for h in ablation_histories if h["growth_rate"] == baseline_gr and h["depth"] == baseline_depth]
    
    # Depth runs: filter by baseline growth_rate and compression
    depth_runs = [h for h in ablation_histories if h["growth_rate"] == baseline_gr and h["compression"] == baseline_comp]
    
    # Batch size runs: baseline configuration
    batch_size_runs = [h for h in ablation_histories if h["growth_rate"] == baseline_gr and h["compression"] == baseline_comp and h["depth"] == baseline_depth]
    
    # Plot 1: Growth Rate Ablation
    if growth_rate_runs:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        for run in growth_rate_runs:
            hist = run["history"]
            epochs = range(1, len(hist["accuracy"]) + 1)
            label = f"k={run['growth_rate']}, bs={run['batch_size']}"
            
            axes[0].plot(epochs, hist["accuracy"], label=f"{label} (train)", linestyle="-", alpha=0.7)
            axes[0].plot(epochs, hist["val_accuracy"], label=f"{label} (val)", linestyle="--", alpha=0.7)
            axes[1].plot(epochs, hist["loss"], label=f"{label} (train)", linestyle="-", alpha=0.7)
            axes[1].plot(epochs, hist["val_loss"], label=f"{label} (val)", linestyle="--", alpha=0.7)
        
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Accuracy")
        axes[0].set_title("Growth Rate Ablation - Accuracy")
        axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        axes[0].grid(True, alpha=0.3)
        
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Loss")
        axes[1].set_title("Growth Rate Ablation - Loss")
        axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        if save_dir:
            plt.savefig(save_dir / "ablation_growth_rate.png", dpi=300, bbox_inches='tight')
        plt.show()
    
    # Plot 2: Compression Ablation
    if compression_runs:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        for run in compression_runs:
            hist = run["history"]
            epochs = range(1, len(hist["accuracy"]) + 1)
            label = f"c={run['compression']}, bs={run['batch_size']}"
            
            axes[0].plot(epochs, hist["accuracy"], label=f"{label} (train)", linestyle="-", alpha=0.7)
            axes[0].plot(epochs, hist["val_accuracy"], label=f"{label} (val)", linestyle="--", alpha=0.7)
            axes[1].plot(epochs, hist["loss"], label=f"{label} (train)", linestyle="-", alpha=0.7)
            axes[1].plot(epochs, hist["val_loss"], label=f"{label} (val)", linestyle="--", alpha=0.7)
        
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Accuracy")
        axes[0].set_title("Compression Ablation - Accuracy")
        axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        axes[0].grid(True, alpha=0.3)
        
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Loss")
        axes[1].set_title("Compression Ablation - Loss")
        axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        if save_dir:
            plt.savefig(save_dir / "ablation_compression.png", dpi=300, bbox_inches='tight')
        plt.show()
    
    # Plot 3: Depth Ablation
    if depth_runs:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        for run in depth_runs:
            hist = run["history"]
            epochs = range(1, len(hist["accuracy"]) + 1)
            depth_str = '-'.join(map(str, run["depth"]))
            label = f"d={depth_str}, bs={run['batch_size']}"
            
            axes[0].plot(epochs, hist["accuracy"], label=f"{label} (train)", linestyle="-", alpha=0.7)
            axes[0].plot(epochs, hist["val_accuracy"], label=f"{label} (val)", linestyle="--", alpha=0.7)
            axes[1].plot(epochs, hist["loss"], label=f"{label} (train)", linestyle="-", alpha=0.7)
            axes[1].plot(epochs, hist["val_loss"], label=f"{label} (val)", linestyle="--", alpha=0.7)
        
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Accuracy")
        axes[0].set_title("Depth Ablation - Accuracy")
        axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        axes[0].grid(True, alpha=0.3)
        
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Loss")
        axes[1].set_title("Depth Ablation - Loss")
        axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        if save_dir:
            plt.savefig(save_dir / "ablation_depth.png", dpi=300, bbox_inches='tight')
        plt.show()
    
    # Plot 4: Batch Size Ablation (baseline config)
    if batch_size_runs:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        for run in batch_size_runs:
            hist = run["history"]
            epochs = range(1, len(hist["accuracy"]) + 1)
            label = f"bs={run['batch_size']}"
            
            axes[0].plot(epochs, hist["accuracy"], label=f"{label} (train)", linestyle="-", alpha=0.7)
            axes[0].plot(epochs, hist["val_accuracy"], label=f"{label} (val)", linestyle="--", alpha=0.7)
            axes[1].plot(epochs, hist["loss"], label=f"{label} (train)", linestyle="-", alpha=0.7)
            axes[1].plot(epochs, hist["val_loss"], label=f"{label} (val)", linestyle="--", alpha=0.7)
        
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Accuracy")
        axes[0].set_title("Batch Size Ablation - Accuracy (k=8, c=0.5, d=3-3-3)")
        axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        axes[0].grid(True, alpha=0.3)
        
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Loss")
        axes[1].set_title("Batch Size Ablation - Loss (k=8, c=0.5, d=3-3-3)")
        axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        if save_dir:
            plt.savefig(save_dir / "ablation_batch_size.png", dpi=300, bbox_inches='tight')
        plt.show()


def plot_combined_ablation_summary(ablation_histories, save_dir=None):
    """
    Create a comprehensive summary plot showing all ablation results.
    """
    if save_dir:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
    
    # Create a large figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Group runs (using baseline values)
    baseline_comp = 0.5
    baseline_depth = (3,3,3)
    baseline_gr = 8
    
    growth_rate_runs = [h for h in ablation_histories if h["compression"] == baseline_comp and h["depth"] == baseline_depth]
    compression_runs = [h for h in ablation_histories if h["growth_rate"] == baseline_gr and h["depth"] == baseline_depth]
    depth_runs = [h for h in ablation_histories if h["growth_rate"] == baseline_gr and h["compression"] == baseline_comp]
    batch_size_runs = [h for h in ablation_histories if h["growth_rate"] == baseline_gr and h["compression"] == baseline_comp and h["depth"] == baseline_depth]
    
    # Top-left: Growth Rate
    ax = axes[0, 0]
    for run in growth_rate_runs:
        hist = run["history"]
        epochs = range(1, len(hist["accuracy"]) + 1)
        ax.plot(epochs, hist["val_accuracy"], label=f"k={run['growth_rate']}, bs={run['batch_size']}", linewidth=2)
    ax.set_xlabel("Epoch", fontsize=10)
    ax.set_ylabel("Validation Accuracy", fontsize=10)
    ax.set_title("Growth Rate Ablation", fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Top-right: Compression
    ax = axes[0, 1]
    for run in compression_runs:
        hist = run["history"]
        epochs = range(1, len(hist["accuracy"]) + 1)
        ax.plot(epochs, hist["val_accuracy"], label=f"c={run['compression']}, bs={run['batch_size']}", linewidth=2)
    ax.set_xlabel("Epoch", fontsize=10)
    ax.set_ylabel("Validation Accuracy", fontsize=10)
    ax.set_title("Compression Ablation", fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Bottom-left: Depth
    ax = axes[1, 0]
    for run in depth_runs:
        hist = run["history"]
        epochs = range(1, len(hist["accuracy"]) + 1)
        depth_str = '-'.join(map(str, run["depth"]))
        ax.plot(epochs, hist["val_accuracy"], label=f"d={depth_str}, bs={run['batch_size']}", linewidth=2)
    ax.set_xlabel("Epoch", fontsize=10)
    ax.set_ylabel("Validation Accuracy", fontsize=10)
    ax.set_title("Depth Ablation", fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Bottom-right: Batch Size
    ax = axes[1, 1]
    for run in batch_size_runs:
        hist = run["history"]
        epochs = range(1, len(hist["accuracy"]) + 1)
        ax.plot(epochs, hist["val_accuracy"], label=f"bs={run['batch_size']}", linewidth=2)
    ax.set_xlabel("Epoch", fontsize=10)
    ax.set_ylabel("Validation Accuracy", fontsize=10)
    ax.set_title("Batch Size Ablation", fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    plt.suptitle("Ablation Study Summary - Validation Accuracy", fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    if save_dir:
        plt.savefig(save_dir / "ablation_summary_accuracy.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    # Loss summary
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Top-left: Growth Rate Loss
    ax = axes[0, 0]
    for run in growth_rate_runs:
        hist = run["history"]
        epochs = range(1, len(hist["loss"]) + 1)
        ax.plot(epochs, hist["val_loss"], label=f"k={run['growth_rate']}, bs={run['batch_size']}", linewidth=2)
    ax.set_xlabel("Epoch", fontsize=10)
    ax.set_ylabel("Validation Loss", fontsize=10)
    ax.set_title("Growth Rate Ablation", fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Top-right: Compression Loss
    ax = axes[0, 1]
    for run in compression_runs:
        hist = run["history"]
        epochs = range(1, len(hist["loss"]) + 1)
        ax.plot(epochs, hist["val_loss"], label=f"c={run['compression']}, bs={run['batch_size']}", linewidth=2)
    ax.set_xlabel("Epoch", fontsize=10)
    ax.set_ylabel("Validation Loss", fontsize=10)
    ax.set_title("Compression Ablation", fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Bottom-left: Depth Loss
    ax = axes[1, 0]
    for run in depth_runs:
        hist = run["history"]
        epochs = range(1, len(hist["loss"]) + 1)
        depth_str = '-'.join(map(str, run["depth"]))
        ax.plot(epochs, hist["val_loss"], label=f"d={depth_str}, bs={run['batch_size']}", linewidth=2)
    ax.set_xlabel("Epoch", fontsize=10)
    ax.set_ylabel("Validation Loss", fontsize=10)
    ax.set_title("Depth Ablation", fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Bottom-right: Batch Size Loss
    ax = axes[1, 1]
    for run in batch_size_runs:
        hist = run["history"]
        epochs = range(1, len(hist["loss"]) + 1)
        ax.plot(epochs, hist["val_loss"], label=f"bs={run['batch_size']}", linewidth=2)
    ax.set_xlabel("Epoch", fontsize=10)
    ax.set_ylabel("Validation Loss", fontsize=10)
    ax.set_title("Batch Size Ablation", fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    plt.suptitle("Ablation Study Summary - Validation Loss", fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    if save_dir:
        plt.savefig(save_dir / "ablation_summary_loss.png", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# =========================
# 11) Generate Ablation Study Plots
# =========================
# Plot individual ablation studies
print("Generating ablation study plots...")
plot_ablation_accuracy_loss(ablation_histories, save_dir=RUNS_DIR / "plots")

# Plot combined summary
print("Generating combined summary plots...")
plot_combined_ablation_summary(ablation_histories, save_dir=RUNS_DIR / "plots")

print("All plots saved to runs/plots/")

In [ ]:
# 9A) Resolution sweep: 32x32, 64x64, 128x128
resolutions = [32, 64, 128]
for r in resolutions:
    run_dir, row, history = train_eval_one_run(
        datadir=DATADIR,
        categories=CATEGORIES,
        img_res=r,
        epochs=BASE_EPOCHS,
        batch_size=BASE_BATCH,
        max_images_per_class=MAX_PER_CLASS,
        **baseline_params
    )

print("Resolution sweep complete. Aggregates at runs/_aggregate_metrics.csv")


In [ ]:
# 9B) Ablation study: growth rate, compression, depth, and batch size (fixed resolution = 64)
# Comprehensive ablation with varied combinations
import itertools

# Define parameter ranges for comprehensive ablation
growth_rates = [4, 6, 8, 10, 12, 16]
compressions = [0.25, 0.35, 0.5, 0.65, 0.75]
depths = [(2,2,2), (2,3,3), (3,3,3), (3,4,4), (4,4,4)]

# Generate systematic combinations
# 1. Full factorial for baseline (growth_rate=8, compression=0.5, depth=(3,3,3)) variations
# 2. One-at-a-time variations (keeping others at baseline)
# 3. Selected interesting combinations

baseline = {"growth_rate": 8, "compression": 0.5, "depth": (3,3,3)}

ablation_grid = []

# One-at-a-time variations (keeping others at baseline)
# Growth rate sweep
for gr in growth_rates:
    ablation_grid.append({"growth_rate": gr, "compression": baseline["compression"], "depth": baseline["depth"]})

# Compression sweep
for comp in compressions:
    ablation_grid.append({"growth_rate": baseline["growth_rate"], "compression": comp, "depth": baseline["depth"]})

# Depth sweep
for depth in depths:
    ablation_grid.append({"growth_rate": baseline["growth_rate"], "compression": baseline["compression"], "depth": depth})

# Selected combinations: growth_rate × compression (with baseline depth)
for gr, comp in itertools.product([4, 8, 12, 16], [0.35, 0.5, 0.65]):
    if not (gr == baseline["growth_rate"] and comp == baseline["compression"]):  # Avoid duplicate
        ablation_grid.append({"growth_rate": gr, "compression": comp, "depth": baseline["depth"]})

# Selected combinations: growth_rate × depth (with baseline compression)
for gr, depth in itertools.product([4, 8, 12], [(2,2,2), (3,3,3), (4,4,4)]):
    if not (gr == baseline["growth_rate"] and depth == baseline["depth"]):  # Avoid duplicate
        ablation_grid.append({"growth_rate": gr, "compression": baseline["compression"], "depth": depth})

# Selected combinations: compression × depth (with baseline growth_rate)
for comp, depth in itertools.product([0.35, 0.5, 0.65], [(2,2,2), (3,3,3), (4,4,4)]):
    if not (comp == baseline["compression"] and depth == baseline["depth"]):  # Avoid duplicate
        ablation_grid.append({"growth_rate": baseline["growth_rate"], "compression": comp, "depth": depth})

# Key 3-way combinations (growth_rate × compression × depth)
key_combinations = [
    (4, 0.35, (2,2,2)),
    (4, 0.65, (4,4,4)),
    (12, 0.35, (2,2,2)),
    (12, 0.65, (4,4,4)),
    (16, 0.5, (3,3,3)),
    (6, 0.5, (3,3,3)),
    (10, 0.5, (3,3,3)),
    (8, 0.25, (3,3,3)),
    (8, 0.75, (3,3,3)),
    (8, 0.5, (2,3,3)),
    (8, 0.5, (3,4,4)),
]
for gr, comp, depth in key_combinations:
    ablation_grid.append({"growth_rate": gr, "compression": comp, "depth": depth})

# Remove duplicates while preserving order
seen = set()
unique_grid = []
for cfg in ablation_grid:
    key = (cfg["growth_rate"], cfg["compression"], cfg["depth"])
    if key not in seen:
        seen.add(key)
        unique_grid.append(cfg)
ablation_grid = unique_grid

print(f"Total ablation configurations: {len(ablation_grid)}")
print(f"Growth rates: {sorted(set(c['growth_rate'] for c in ablation_grid))}")
print(f"Compressions: {sorted(set(c['compression'] for c in ablation_grid))}")
print(f"Depths: {sorted(set(c['depth'] for c in ablation_grid))}")

# Batch sizes to test
batch_sizes = [32, 64, 128, 256]

print(f"\nBatch sizes to test: {batch_sizes}")
print(f"Total runs to execute: {len(ablation_grid)} configs × {len(batch_sizes)} batch sizes = {len(ablation_grid) * len(batch_sizes)} runs")
print("\nStarting ablation study...\n")

ablation_csv = RUNS_DIR / "_ablation.csv"
if ablation_csv.exists():
    ablation_csv.unlink()

# Store all run histories for plotting
ablation_histories = []

for cfg in ablation_grid:
    for batch_size in batch_sizes:
        run_dir, row, history = train_eval_one_run(
            datadir=DATADIR,
            categories=CATEGORIES,
            img_res=64,
            epochs=BASE_EPOCHS,
            batch_size=batch_size,
            max_images_per_class=MAX_PER_CLASS,
            **cfg
        )
        ablation_row = {
            "run_id": row["run_id"],
            "img_res": row["img_res"],
            "growth_rate": row["growth_rate"],
            "compression": row["compression"],
            "depth": row["depth"],
            "batch_size": row["batch_size"],
            "accuracy": row["accuracy"],
            "macro_precision": row["macro_precision"],
            "macro_recall": row["macro_recall"],
            "macro_f1": row["macro_f1"],
            "avg_latency_ms": row["avg_latency_ms"],
            "fps": row["fps"],
        }
        append_row_csv(ablation_csv, ablation_row)
        
        # Store history for plotting
        ablation_histories.append({
            "run_id": row["run_id"],
            "growth_rate": cfg["growth_rate"],
            "compression": cfg["compression"],
            "depth": cfg["depth"],
            "batch_size": batch_size,
            "history": history.history
        })

print(f"Ablation complete. Total runs: {len(ablation_histories)}. Table-ready: runs/_ablation.csv")


## Output summary
- Per run: `runs/<run_id>/{config.json, device.json, metrics.json, confusion_matrix.json, confusion_matrix.csv, latency.json, history.csv}`
- Aggregates: `runs/_aggregate_metrics.csv` (Table IV) and `runs/_ablation.csv` (Ablation Table)

If a run fails, it will fail fast with a clear error message (missing folders, no images loaded, etc.), rather than silently corrupting results.